# Week 6 Assignment: Price Band Classification Fine-tuning

## Task
Predict price class (budget, midrange, premium) instead of exact price.

## Assignment Goals:
1. Create price bins (budget, midrange, premium)
2. Fine-tune gpt-4o-mini-2024-07-18 for classification
3. Report F1 score and classification metrics

## Steps:
- Create price bands based on data distribution
- Prepare fine-tuning data in JSONL format
- Fine-tune the model on classification task
- Evaluate with F1 score and classification report

## 1. Imports and Setup

In [ ]:
# Core imports
import os
import json
import random
import time
from dotenv import load_dotenv

# Load environment variables first
load_dotenv(override=True)

# HuggingFace and OpenAI
from huggingface_hub import login
from openai import OpenAI

# Data analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Machine Learning imports
from sklearn.metrics import classification_report, f1_score, confusion_matrix, accuracy_score

# Week 6 imports - import Item class from pricer module
import sys
sys.path.append('.')
from pricer.items import Item

print("All imports successful")

In [ ]:
# Login to HuggingFace
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

# Initialize OpenAI client
openai_client = OpenAI()

## 2. Configuration

In [ ]:
# Configuration
LITE_MODE = False  # Set to True for faster testing with smaller dataset
HF_USERNAME = "sirmuelemmanuel"

# Fine-tuning settings - start small to minimize cost
TRAIN_SAMPLES = 500  # Recommended: 500-1000 for good performance
VAL_SAMPLES = 100
TEST_SAMPLES = 200

# Random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 3. Load Data from Week 6

In [ ]:
# Load preprocessed data from week 6
username = "ed-donner"  # Using the instructor's dataset
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items")
print(f"Loaded {len(val):,} validation items")
print(f"Loaded {len(test):,} test items")
print(f"\nSample item: {train[0]}")

## 4. Analyze Price Distribution and Create Bins

In [ ]:
# Analyze price distribution
all_items = train + val + test
all_prices = [item.price for item in all_items]

print("Price Statistics:")
print(f"Min: ${min(all_prices):.2f}")
print(f"Max: ${max(all_prices):.2f}")
print(f"Mean: ${np.mean(all_prices):.2f}")
print(f"Median: ${np.median(all_prices):.2f}")
print(f"25th percentile: ${np.percentile(all_prices, 25):.2f}")
print(f"75th percentile: ${np.percentile(all_prices, 75):.2f}")

# Plot price distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(all_prices, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.title('Price Distribution')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(all_prices, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Price ($)')
plt.ylabel('Frequency (log scale)')
plt.title('Price Distribution (Log Scale)')
plt.yscale('log')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Define price bands using percentiles for balanced classes
# We'll use 33rd and 67th percentiles for roughly equal distribution

percentile_33 = np.percentile(all_prices, 33.33)
percentile_67 = np.percentile(all_prices, 66.67)

# Define bins
BUDGET_MAX = percentile_33
MIDRANGE_MAX = percentile_67

print(f"\nPrice Band Definitions:")
print(f"Budget: $0 - ${BUDGET_MAX:.2f}")
print(f"Midrange: ${BUDGET_MAX:.2f} - ${MIDRANGE_MAX:.2f}")
print(f"Premium: ${MIDRANGE_MAX:.2f}+")


def get_price_band(price: float) -> str:
    """Classify price into budget, midrange, or premium"""
    if price <= BUDGET_MAX:
        return "budget"
    elif price <= MIDRANGE_MAX:
        return "midrange"
    else:
        return "premium"


# Test the function
print(f"\nTest classifications:")
test_prices = [10, 50, 100, 200]
for price in test_prices:
    print(f"${price:.2f} -> {get_price_band(price)}")

In [ ]:
# Analyze class distribution
bands = [get_price_band(price) for price in all_prices]
band_counts = pd.Series(bands).value_counts()

print("\nClass Distribution:")
for band, count in band_counts.items():
    percentage = (count / len(bands)) * 100
    print(f"{band.capitalize()}: {count:,} ({percentage:.1f}%)")

# Visualize class distribution
plt.figure(figsize=(8, 5))
band_counts.plot(kind='bar', color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.7)
plt.xlabel('Price Band')
plt.ylabel('Count')
plt.title('Distribution of Price Bands')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Prepare Fine-tuning Data

In [ ]:
# Create dictionaries mapping items to their price bands
# Since Item is a Pydantic model, we can't add fields dynamically
def create_items_with_bands(items):
    """Create list of (item, price_band) tuples"""
    return [(item, get_price_band(item.price)) for item in items]

train_with_bands = create_items_with_bands(train)
val_with_bands = create_items_with_bands(val)
test_with_bands = create_items_with_bands(test)

print("Sample items with price bands:")
for i in range(3):
    item, price_band = train_with_bands[i]
    print(f"{i+1}. {item.title[:50]}... | ${item.price:.2f} | {price_band}")

In [ ]:
# Create balanced subsets for fine-tuning
def create_balanced_subset(items_with_bands, n_samples):
    """Create a balanced subset with equal representation from each class"""
    # Group items by price band
    budget_items = [(item, band) for item, band in items_with_bands if band == "budget"]
    midrange_items = [(item, band) for item, band in items_with_bands if band == "midrange"]
    premium_items = [(item, band) for item, band in items_with_bands if band == "premium"]
    
    # Calculate samples per class (roughly equal)
    samples_per_class = n_samples // 3
    
    # Shuffle and sample
    random.shuffle(budget_items)
    random.shuffle(midrange_items)
    random.shuffle(premium_items)
    
    subset = (
        budget_items[:samples_per_class] +
        midrange_items[:samples_per_class] +
        premium_items[:samples_per_class]
    )
    
    random.shuffle(subset)
    return subset


# Create subsets
fine_tune_train = create_balanced_subset(train_with_bands, TRAIN_SAMPLES)
fine_tune_val = create_balanced_subset(val_with_bands, VAL_SAMPLES)
fine_tune_test = create_balanced_subset(test_with_bands, TEST_SAMPLES)

print(f"\nCreated fine-tuning datasets:")
print(f"Train: {len(fine_tune_train)} samples")
print(f"Validation: {len(fine_tune_val)} samples")
print(f"Test: {len(fine_tune_test)} samples")

# Verify class balance
train_bands = [band for item, band in fine_tune_train]
print(f"\nTraining set class distribution:")
for band in ['budget', 'midrange', 'premium']:
    count = train_bands.count(band)
    print(f"{band.capitalize()}: {count} ({count/len(train_bands)*100:.1f}%)")

In [ ]:
# Create message format for fine-tuning
def messages_for_classification(item, price_band):
    """Create messages for price band classification task"""
    message = (
        f"Classify the price band of this product as 'budget', 'midrange', or 'premium'. "
        f"Respond with only the price band, no explanation.\n\n{item.summary}"
    )
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": price_band}
    ]


# Test the function
print("Sample training message:")
sample_item, sample_band = fine_tune_train[0]
sample_messages = messages_for_classification(sample_item, sample_band)
for msg in sample_messages:
    print(f"\n{msg['role'].upper()}:")
    print(msg['content'][:200] + "..." if len(msg['content']) > 200 else msg['content'])

In [ ]:
# Save training and validation data in JSONL format for OpenAI fine-tuning
def save_jsonl(items_with_bands, filename):
    """Save items to JSONL file in OpenAI format"""
    with open(filename, 'w') as f:
        for item, price_band in items_with_bands:
            json_obj = {"messages": messages_for_classification(item, price_band)}
            f.write(json.dumps(json_obj) + '\n')
    print(f"Saved {len(items_with_bands)} items to {filename}")


# Save files
train_file = "fine_tune_classification_train.jsonl"
val_file = "fine_tune_classification_validation.jsonl"

save_jsonl(fine_tune_train, train_file)
save_jsonl(fine_tune_val, val_file)

## 6. Upload Files to OpenAI

In [ ]:
# Upload training file
print("Uploading training file...")
with open(train_file, "rb") as f:
    train_file_obj = openai_client.files.create(
        file=f,
        purpose="fine-tune"
    )

print(f"Training file uploaded: {train_file_obj.id}")

# Upload validation file
print("\nUploading validation file...")
with open(val_file, "rb") as f:
    val_file_obj = openai_client.files.create(
        file=f,
        purpose="fine-tune"
    )

print(f"Validation file uploaded: {val_file_obj.id}")

# Store file IDs for later use
train_file_id = train_file_obj.id
val_file_id = val_file_obj.id

## 7. Create Fine-tuning Job

In [ ]:
# Create fine-tuning job
print("Creating fine-tuning job...")

fine_tune_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file_id,
    validation_file=val_file_id,
    model="gpt-4o-mini-2024-07-18",
    suffix="price-band-classifier",
    hyperparameters={
        "n_epochs": 3  # Adjust based on performance
    }
)

job_id = fine_tune_job.id
print(f"\nFine-tuning job created: {job_id}")
print(f"Status: {fine_tune_job.status}")
print(f"Model: {fine_tune_job.model}")

# Save job ID for later reference
with open('fine_tune_job_id.txt', 'w') as f:
    f.write(job_id)
print(f"\nJob ID saved to fine_tune_job_id.txt")

## 8. Monitor Fine-tuning Job

In [ ]:
job_id = "ftjob-CPZ5XESMudnEMHpLEYpKRHvI"
# Monitor job progress
def check_job_status(job_id):
    """Check the status of a fine-tuning job"""
    job = openai_client.fine_tuning.jobs.retrieve(job_id)
    print(f"Status: {job.status}")
    if job.status == "succeeded":
        print(f"Fine-tuned model: {job.fine_tuned_model}")
        return job.fine_tuned_model
    elif job.status == "failed":
        print(f"Error: {job.error}")
    return None


# Check current status
print("Current job status:")
model_id = check_job_status(job_id)

# Wait for completion (optional - can run this cell multiple times)
print("\nWaiting for job to complete...")
print("This may take 10-30 minutes. You can check status by re-running this cell.")

while model_id is None:
    time.sleep(60)  # Check every minute
    job = openai_client.fine_tuning.jobs.retrieve(job_id)
    status = job.status
    print(f"Status: {status} - {time.strftime('%H:%M:%S')}")
    
    if status == "succeeded":
        model_id = job.fine_tuned_model
        print(f"\nFine-tuning completed!")
        print(f"Fine-tuned model: {model_id}")
        break
    elif status == "failed":
        print(f"✗ Fine-tuning failed: {job.error}")
        break

# Save model ID
if model_id:
    with open('fine_tuned_model_id.txt', 'w') as f:
        f.write(model_id)
    print(f"\nModel ID saved to fine_tuned_model_id.txt")

## 9. Load Fine-tuned Model (After Job Completes)

In [ ]:
# If you're running this later, load the model ID from file
try:
    with open('fine_tuned_model_id.txt', 'r') as f:
        model_id = f.read().strip()
    print(f"Loaded model ID: {model_id}")
except FileNotFoundError:
    print("Model ID file not found. Make sure fine-tuning is complete.")
    model_id = None

## 10. Evaluate the Fine-tuned Model

In [ ]:
# Create prediction function
def predict_price_band(item, model_id):
    """Predict price band for an item using the fine-tuned model"""
    message = (
        f"Classify the price band of this product as 'budget', 'midrange', or 'premium'. "
        f"Respond with only the price band, no explanation.\n\n{item.summary}"
    )
    
    try:
        response = openai_client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "user", "content": message}
            ],
            temperature=0,  # Use 0 for consistent predictions
            max_tokens=10
        )
        
        prediction = response.choices[0].message.content.strip().lower()
        
        # Ensure prediction is one of the valid classes
        if prediction not in ['budget', 'midrange', 'premium']:
            # Try to extract the class from the response
            for band in ['budget', 'midrange', 'premium']:
                if band in prediction:
                    return band
            return 'midrange'  # Default fallback
        
        return prediction
    
    except Exception as e:
        print(f"Error predicting: {e}")
        return 'midrange'  # Default fallback


# Test on a single item
if model_id:
    test_item, test_band = fine_tune_test[0]
    prediction = predict_price_band(test_item, model_id)
    print(f"Test Item: {test_item.title[:50]}...")
    print(f"Actual price: ${test_item.price:.2f} ({test_band})")
    print(f"Predicted: {prediction}")
else:
    print("Model ID not available. Complete fine-tuning first.")

In [ ]:
# Make predictions on test set
if model_id:
    print(f"Making predictions on {len(fine_tune_test)} test items...")
    print("This may take a few minutes.\n")
    
    predictions = []
    true_labels = []
    
    for i, (item, true_band) in enumerate(fine_tune_test):
        if (i + 1) % 20 == 0:
            print(f"Progress: {i + 1}/{len(fine_tune_test)}")
        
        pred = predict_price_band(item, model_id)
        predictions.append(pred)
        true_labels.append(true_band)
        
        # Add small delay to avoid rate limits
        time.sleep(0.1)
    
    print(f"\nCompleted {len(predictions)} predictions")
else:
    print("Model ID not available. Complete fine-tuning first.")

## 11. Classification Report and Metrics

In [ ]:
# Calculate F1 scores
if model_id and len(predictions) > 0:
    # Overall metrics
    accuracy = accuracy_score(true_labels, predictions)
    f1_micro = f1_score(true_labels, predictions, average='micro')
    f1_macro = f1_score(true_labels, predictions, average='macro')
    f1_weighted = f1_score(true_labels, predictions, average='weighted')
    
    print("="*60)
    print("OVERALL PERFORMANCE METRICS")
    print("="*60)
    print(f"Accuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"F1 Score (Micro):  {f1_micro:.4f}")
    print(f"F1 Score (Macro):  {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    print("="*60)
else:
    print("No predictions available. Complete evaluation first.")

In [ ]:
# Detailed classification report
if model_id and len(predictions) > 0:
    print("\n" + "="*60)
    print("DETAILED CLASSIFICATION REPORT")
    print("="*60)
    report = classification_report(
        true_labels, 
        predictions,
        labels=['budget', 'midrange', 'premium'],
        target_names=['Budget', 'Midrange', 'Premium'],
        digits=4
    )
    print(report)
    
    # Save report to file
    with open('classification_report.txt', 'w') as f:
        f.write("PRICE BAND CLASSIFICATION RESULTS\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Model: {model_id}\n")
        f.write(f"Test samples: {len(predictions)}\n\n")
        f.write("OVERALL METRICS:\n")
        f.write(f"Accuracy: {accuracy:.4f}\n")
        f.write(f"F1 Score (Macro): {f1_macro:.4f}\n")
        f.write(f"F1 Score (Weighted): {f1_weighted:.4f}\n\n")
        f.write("DETAILED REPORT:\n")
        f.write(report)
    
    print("\nReport saved to classification_report.txt")
else:
    print("No predictions available. Complete evaluation first.")

In [ ]:
# Confusion Matrix
if model_id and len(predictions) > 0:
    cm = confusion_matrix(true_labels, predictions, labels=['budget', 'midrange', 'premium'])
    
    # Create confusion matrix heatmap using matplotlib only
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create heatmap
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Count', rotation=270, labelpad=20)
    
    # Set ticks and labels
    classes = ['Budget', 'Midrange', 'Premium']
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    
    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=0, ha='center')
    
    # Add text annotations
    for i in range(len(classes)):
        for j in range(len(classes)):
            text = ax.text(j, i, str(cm[i, j]),
                          ha='center', va='center', color='black' if cm[i, j] < cm.max()/2 else 'white',
                          fontsize=14, fontweight='bold')
    
    ax.set_title('Confusion Matrix - Price Band Classification', fontsize=14, pad=20)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Confusion matrix saved to confusion_matrix.png")
else:
    print("No predictions available. Complete evaluation first.")

## 12. Per-Class F1 Scores

In [ ]:
# Calculate F1 score for each class
if model_id and len(predictions) > 0:
    f1_per_class = f1_score(true_labels, predictions, labels=['budget', 'midrange', 'premium'], average=None)
    
    print("\n" + "="*60)
    print("F1 SCORES BY PRICE BAND")
    print("="*60)
    
    classes = ['Budget', 'Midrange', 'Premium']
    for i, (cls, score) in enumerate(zip(classes, f1_per_class)):
        print(f"{cls:12} F1: {score:.4f} ({score*100:.2f}%)")
    
    print("="*60)
    
    # Visualize F1 scores
    plt.figure(figsize=(10, 6))
    bars = plt.bar(classes, f1_per_class, color=['#3498db', '#2ecc71', '#e74c3c'], alpha=0.7)
    plt.xlabel('Price Band', fontsize=12)
    plt.ylabel('F1 Score', fontsize=12)
    plt.title('F1 Scores by Price Band', fontsize=14)
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, score in zip(bars, f1_per_class):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{score:.4f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('f1_scores_by_class.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nF1 scores visualization saved to f1_scores_by_class.png")
else:
    print("No predictions available. Complete evaluation first.")